In [1]:
platformID = 'INS'

In [2]:
from datetime import datetime, date
import pandas as pd
import numpy as np
import os 

import psycopg2
import missingno as msno

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import lookup_loader, execute_sql_query, compare_or_update_reference, fix_country_one_percent_prev_week
import test_functions 

In [4]:
lookup = lookup_loader(gam_info, platformID, '1c',
                       with_country=True, country_col=['YT-_FBE_codes', 'ins_country_name',])
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


✅ Test INS_1c_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test INS_1c_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test INS_1c_02 passed: No empty values in lookup.
...updating logbook...

✅ Test INS_1c_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test INS_1c_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test INS_1c_05 passed: No empty values in lookup.
...updating logbook...

✅ Test INS_1c_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test INS_1c_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test INS_1c_08 passed: No empty values in lookup.
...updating logbook...



In [5]:
country_codes[country_codes['ins_country_name'] == 'Iran']

,ins_country_name,PlaceID,YT-_FBE_codes
101,Iran,IRN,IR


# ingestion

In [6]:
# keep country code where clause - the other rows are for age & gender in diff cols
sql_query = f"""
    SELECT 
        week_commencing,
        l.ig_account_id as page_id,
        page_name, 
        country_code,
        country_name as ins_country_name,
        followers_by_demographic
    FROM
        redshiftdb.central_insights.adverity_social_instagram_by_page_demo AS p
    RIGHT JOIN
            world_service_audiences_insights.social_media_lookup_ig AS l
        ON 
            p.page_id = l.ig_identifier
    WHERE
            week_commencing Between '{gam_info['w/c_start']}' and '{gam_info['w/c_end']}'
        AND
            followers_by_demographic > 0
        AND 
            country_name <> ''
        ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+df['page_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

ig_userCountry_raw = pd.read_csv(file, keep_default_na=False)

ig_userCountry_raw['page_id'] = ig_userCountry_raw['page_id'].astype(str) 
ig_userCountry_raw['week_commencing'] = pd.to_datetime(ig_userCountry_raw['week_commencing'])
ig_userCountry_raw = ig_userCountry_raw.rename(columns={'page_id': 'Channel ID',
                                                'page_name': 'Channel Name',
                                                'week_commencing': 'w/c',
                                                'country_code': 'YT-_FBE_codes',
                                                })


ig_userCountry_raw = ig_userCountry_raw.merge(country_codes[['YT-_FBE_codes', 'ins_country_name']], 
                                          on='ins_country_name', suffixes=['', '_name_based'],
                                          how='left').drop(columns=['ins_country_name'])
ig_userCountry_raw.loc[ig_userCountry_raw['YT-_FBE_codes'] == '', 'YT-_FBE_codes'] = ig_userCountry_raw.loc[ig_userCountry_raw['YT-_FBE_codes'] == '', 
    'YT-_FBE_codes_name_based']

ig_userCountry_raw = ig_userCountry_raw.drop(columns=['YT-_FBE_codes_name_based']).drop_duplicates()

### RUN TESTS
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
# missing page_ids
test_functions.test_filter_elements_returned(ig_userCountry_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_09",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=ig_userCountry_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(ig_userCountry_raw, 
                           numeric_columns=['followers_by_demographic'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in followers_by_demographic column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(ig_userCountry_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')



no redshift connection using last time queried
...testing Channel ID...
✅ Test INS_1c_09 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
                Channel ID      Start End        w/c
1638  INS17841402856323858 2020-04-01 NaT 2025-03-31
2394  INS17841435042594492 2020-04-01 NaT 2025-03-31
2772  INS17841473849472688 2020-04-01 NaT 2025-03-31
2016  INS17841405843432881 2020-04-01 NaT 2025-03-31
2646  INS17841464755735820 2020-04-01 NaT 2025-03-31
...                    ...        ...  ..        ...
2431  INS17841435042594492 2020-04-01 NaT 2025-12-15
2432  INS17841435042594492 2020-04-01 NaT 2025-12-22
2433  INS17841435042594492 2020-04-01 NaT 2025-12-29
2434  INS17841435042594492 2020-04-01 NaT 2026-01-05
2435  INS17841435042594492 2020-04-01 NaT 2026-01-12

[147 rows x 4 columns]

Summary of missing weeks per channel_id_col:
              Channel ID  missing_week_count
57  INS17841435042594492                  40
66  INS17841473849472688      

In [7]:
test_functions.test_inner_join(ig_userCountry_raw, socialmedia_accounts, 
                               ['Channel ID'], 
                               f"{platformID}_1c_13",
                               test_step='checking social media accounts in lookup, adding service',
                               focus='left')
ig_userCountry = ig_userCountry_raw.merge(socialmedia_accounts[['Channel ID', 'ServiceID']], 
                                                      on='Channel ID', how='left')

✅ Inner join test INS_1c_13 successful: No issues found.
...updating logbook...



In [8]:
test_functions.test_inner_join(ig_userCountry, 
                               country_codes, 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_14",
                               test_step='calculating country %', 
                                focus='left')
ig_userCountry = ig_userCountry.merge(country_codes[['YT-_FBE_codes', 'PlaceID']], 
                                          on='YT-_FBE_codes', 
                                          how='left').drop(columns=['YT-_FBE_codes'])

✅ Inner join test INS_1c_14 successful: No issues found.
...updating logbook...



In [9]:
# currently sql table has duplicates
ig_userCountry = ig_userCountry.drop_duplicates()
# test for duplicate entries 
test_functions.test_duplicates(ig_userCountry, ['Channel ID', 'w/c', 'PlaceID'], 
                               test_number=f"{platformID}_1c_15",
                               test_step='dropped duplicates past processing')


✅ Test INS_1c_15 passed: No combinations occurs more than once a week.
...updating logbook...



In [10]:
grouped_df = ig_userCountry.groupby(['Channel ID', 'Channel Name', 
                                     'w/c']).agg({'followers_by_demographic': 'sum'}).reset_index()

# Rename the aggregated column
grouped_df = grouped_df.rename(columns={'followers_by_demographic': 'Sum_followers_by_demographic'})

right_cols = ['Channel ID', 'w/c', 'Sum_followers_by_demographic']
ig_country_df = ig_userCountry.merge(grouped_df[right_cols],
                                         how='inner',
                                         on=['Channel ID', 'w/c'])

test_functions.test_inner_join(ig_userCountry, grouped_df[right_cols], 
                               ['Channel ID', 'w/c'],
                               f"{platformID}_1c_16",)

ig_country_df['country_%'] = ig_country_df.apply(
    lambda row: row['followers_by_demographic'] / row['Sum_followers_by_demographic']
    if row['Sum_followers_by_demographic'] > 0 else 0, axis=1
)

test_functions.test_percentage(ig_country_df, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_17",
                               test_step='summing country % per week & account')

✅ Inner join test INS_1c_16 successful: No issues found.
...updating logbook...

...updating logbook...



In [11]:
ig_country_df.shape

(122952, 8)

In [12]:
'''
### DANGER! FIXING IRAN REACH AT 1% DUE TO CURRENT SHUTDOWN 

def fix_country_one_percent(df, country_code, week_list):
    print(f"Fixing Iran reach at 1% due to shutdown for weeks: {week_list}")
    
    df = df.copy()

    # filter affected weeks ONLY
    affected = df[df['w/c'].isin(set(week_list))]
    print("Affected rows:", affected.shape)

    # loop only over affected weeks
    for (channel, wc), group in affected.groupby(['Channel ID', 'w/c']):
        
        # Iran row
        old_country_pct = group.loc[group['PlaceID'] == country_code, 'country_%']
        if old_country_pct.empty:
            continue

        old_country_pct = float(old_country_pct.iloc[0])
        new_country_pct = 0.01

        if abs(old_country_pct - new_country_pct) < 1e-9:
            continue

        # scale all OTHER countries proportionally
        scale_factor = (1 - new_country_pct) / (1 - old_country_pct)

        group_mask        = (df['Channel ID'] == channel) & (df['w/c'] == wc)
        country_mask      = group_mask & (df['PlaceID'] == country_code)
        non_country_mask  = group_mask & (df['PlaceID'] != country_code)

        df.loc[non_country_mask, 'country_%'] *= scale_factor
        df.loc[country_mask, 'country_%'] = new_country_pct

    return df


# Iran = PlaceID "IRN"
IRAN_CODE = "IRN"

ig_country_df = fix_country_one_percent(
    ig_country_df,
    IRAN_CODE,
    ["2026-01-12"]
)
'''


# Iran = PlaceID "IRN"
IRAN_CODE = "IRN"

ig_country_df = fix_country_one_percent_prev_week(
    ig_country_df,
    IRAN_CODE,
    ["2026-01-12"]
)

Fixing Iran reach to 1% of previous week's value for weeks: ['2026-01-12']
Affected rows: (2970, 8)


/Users/dominiquebruns/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:190: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  affected = df[df['w/c'].isin(set(week_list))]


In [13]:
#sanity checks 
# 1
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['PlaceID'] == 'IRN')].head())
# 2
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['Channel ID'] == 'INS17841402172935746')].sort_values('PlaceID'))
# 3 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['Channel ID'] == 'INS17841402172935746')].sort_values('PlaceID')['country_%'].sum())

#sanity checks - previous weeks are not affected
# 4
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['PlaceID'] == 'IRN')].head())
# 5
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['Channel ID'] == 'INS17841402172935746')].sort_values('PlaceID'))
# 6 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['Channel ID'] == 'INS17841402172935746')].sort_values('PlaceID')['country_%'].sum())

,w/c,Channel ID,Channel Name,followers_by_demographic,ServiceID,PlaceID,Sum_followers_by_demographic,country_%
98929,2026-01-12,INS17841400230391592,BBC NEWS فارسی,16589523,PER,IRN,18695878,0.008870
34511,2026-01-12,INS17841400240564037,BBC Culture,23082,BNO,IRN,342476,0.000686
89488,2026-01-12,INS17841400273561016,BBC News Azərbaycanca,49205,AZE,IRN,192013,0.002571
68949,2026-01-12,INS17841400485756476,bbcnewsafrique,1605,FRE,IRN,233665,0.000069
120907,2026-01-12,INS17841400546813714,Top Gear,148295,WOR,IRN,4569184,0.000324


,w/c,Channel ID,Channel Name,followers_by_demographic,ServiceID,PlaceID,Sum_followers_by_demographic,country_%
102908,2026-01-12,INS17841402172935746,BBC Learning English,57785,ELT,ALG,3933960,0.016258
102930,2026-01-12,INS17841402172935746,BBC Learning English,70549,ELT,ARG,3933960,0.019850
102931,2026-01-12,INS17841402172935746,BBC Learning English,44451,ELT,AUS,3933960,0.012507
102936,2026-01-12,INS17841402172935746,BBC Learning English,29472,ELT,AZE,3933960,0.008292
102897,2026-01-12,INS17841402172935746,BBC Learning English,49483,ELT,BAN,3933960,0.013922
102906,2026-01-12,INS17841402172935746,BBC Learning English,221038,ELT,BRA,3933960,0.062191
102913,2026-01-12,INS17841402172935746,BBC Learning English,23306,ELT,BUR,3933960,0.006557
102911,2026-01-12,INS17841402172935746,BBC Learning English,74454,ELT,CAN,3933960,0.020948
102923,2026-01-12,INS17841402172935746,BBC Learning English,34130,ELT,CHI,3933960,0.009603
102921,2026-01-12,INS17841402172935746,BBC Learning English,29491,ELT,CHL,3933960,0.008298


1.0


,w/c,Channel ID,Channel Name,followers_by_demographic,ServiceID,PlaceID,Sum_followers_by_demographic,country_%
98836,2026-01-05,INS17841400230391592,BBC NEWS فارسی,16466541,PER,IRN,18564283,0.887001
34155,2026-01-05,INS17841400240564037,BBC Culture,22980,BNO,IRN,335200,0.068556
89443,2026-01-05,INS17841400273561016,BBC News Azərbaycanca,48963,AZE,IRN,190447,0.257095
68821,2026-01-05,INS17841400485756476,bbcnewsafrique,1597,FRE,IRN,232232,0.006877
120817,2026-01-05,INS17841400546813714,Top Gear,148317,WOR,IRN,4571971,0.032440


,w/c,Channel ID,Channel Name,followers_by_demographic,ServiceID,PlaceID,Sum_followers_by_demographic,country_%
102863,2026-01-05,INS17841402172935746,BBC Learning English,57636,ELT,ALG,3930886,0.014662
102885,2026-01-05,INS17841402172935746,BBC Learning English,70416,ELT,ARG,3930886,0.017914
102886,2026-01-05,INS17841402172935746,BBC Learning English,44797,ELT,AUS,3930886,0.011396
102891,2026-01-05,INS17841402172935746,BBC Learning English,29450,ELT,AZE,3930886,0.007492
102852,2026-01-05,INS17841402172935746,BBC Learning English,49333,ELT,BAN,3930886,0.012550
102861,2026-01-05,INS17841402172935746,BBC Learning English,220500,ELT,BRA,3930886,0.056094
102868,2026-01-05,INS17841402172935746,BBC Learning English,23320,ELT,BUR,3930886,0.005933
102866,2026-01-05,INS17841402172935746,BBC Learning English,74733,ELT,CAN,3930886,0.019012
102878,2026-01-05,INS17841402172935746,BBC Learning English,34004,ELT,CHI,3930886,0.008650
102876,2026-01-05,INS17841402172935746,BBC Learning English,29450,ELT,CHL,3930886,0.007492


0.9999999999999999


In [14]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
ig_country_df[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_geog.csv", 
                           index=None)
'''
compare_or_update_reference(ig_country_df[cols], 
                            f"../test/refactoring/{platformID}_country_expected.pkl", 
                            cols, update=False)
'''

'\ncompare_or_update_reference(ig_country_df[cols], \n                            f"../test/refactoring/{platformID}_country_expected.pkl", \n                            cols, update=False)\n'